## I did 2A with 2 approaches, one with just raw inference and the accuracy results were near zero, and then one with finetuing and the accuracy increaded to near perfect

In [5]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.metrics import classification_report
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Load CIFAR-10 test set
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False, num_workers=2)

# 2. Load pretrained ResNet-18 (ImageNet weights, no modification)
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1).to(device)
model.eval()

# 3. Run inference
correct_top1, correct_top5, total = 0, 0, 0
all_preds, all_targets = [], []

mem_before = torch.cuda.memory_allocated(device) if device.type == 'cuda' else 0
start = time.time()

with torch.no_grad():
    for inputs, targets in testloader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)

        _, pred = outputs.topk(5, 1, True, True)
        pred = pred.t()
        correct = pred.eq(targets.view(1, -1).expand_as(pred))

        correct_top1 += correct[:1].reshape(-1).float().sum().item()
        correct_top5 += correct[:5].reshape(-1).float().sum().item()
        total += targets.size(0)

        all_preds.extend(pred[0].cpu().numpy())
        all_targets.extend(targets.cpu().numpy())

elapsed = time.time() - start
mem_after = torch.cuda.memory_allocated(device) if device.type == 'cuda' else 0

print(f"Top-1 Accuracy : {100 * correct_top1 / total:.2f}%")
print(f"Top-5 Accuracy : {100 * correct_top5 / total:.2f}%")
print(f"Runtime        : {elapsed:.2f}s")
if device.type == 'cuda':
    print(f"GPU Memory Used: {(mem_after - mem_before) / 1e6:.1f} MB")

# 4. Classification report for first 3 CIFAR-10 classes
# Note: predictions are raw ImageNet indices (0-999); CIFAR-10 targets are 0-9.
# Accuracy is near 0% due to label space mismatch without a class mapping.
print("\nClassification Report (First 3 Classes):")
print(classification_report(all_targets, all_preds, labels=[0, 1, 2],
                            target_names=['plane', 'car', 'bird'], zero_division=0))

/home/suwaraich/CS_343_AI/.venv/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Top-1 Accuracy : 0.00%
Top-5 Accuracy : 0.04%
Runtime        : 4.83s
GPU Memory Used: 0.1 MB

Classification Report (First 3 Classes):
              precision    recall  f1-score   support

       plane       0.00      0.00      0.00    1000.0
         car       0.00      0.00      0.00    1000.0
        bird       0.00      0.00      0.00    1000.0

   micro avg       0.00      0.00      0.00    3000.0
   macro avg       0.00      0.00      0.00    3000.0
weighted avg       0.00      0.00      0.00    3000.0



In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.metrics import classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Load Dataset
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False, num_workers=2)

# 2. Load Pretrained Model and replace final layer for CIFAR-10 (10 classes)
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

# 3. Fine-tune
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

EPOCHS = 5
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for inputs, targets in trainloader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        loss = criterion(model(inputs), targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    scheduler.step()
    print(f"Epoch {epoch+1}/{EPOCHS}  Loss: {running_loss/len(trainloader):.4f}")

# 4. Run Inference
model.eval()
correct_top1, correct_top5, total = 0, 0, 0
all_preds, all_targets = [], []

with torch.no_grad():
    for inputs, targets in testloader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)

        _, pred = outputs.topk(5, 1, True, True)
        pred = pred.t()
        correct = pred.eq(targets.view(1, -1).expand_as(pred))

        correct_top1 += correct[:1].reshape(-1).float().sum().item()
        correct_top5 += correct[:5].reshape(-1).float().sum().item()
        total += targets.size(0)

        all_preds.extend(pred[0].cpu().numpy())
        all_targets.extend(targets.cpu().numpy())

print(f"\nTop-1 Accuracy: {100 * correct_top1 / total:.2f}%")
print(f"Top-5 Accuracy: {100 * correct_top5 / total:.2f}%")

# 5. Classification Report (first 3 CIFAR-10 classes)
print("\nClassification Report (First 3 Classes):")
print(classification_report(all_targets, all_preds, labels=[0, 1, 2],
                            target_names=['plane', 'car', 'bird'], zero_division=0))

/home/suwaraich/CS_343_AI/.venv/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Epoch 1/5  Loss: 0.4004
Epoch 2/5  Loss: 0.1713
Epoch 3/5  Loss: 0.0950
Epoch 4/5  Loss: 0.0299
Epoch 5/5  Loss: 0.0132

Top-1 Accuracy: 95.50%
Top-5 Accuracy: 99.92%

Classification Report (First 3 Classes):
              precision    recall  f1-score   support

       plane       0.95      0.97      0.96      1000
         car       0.98      0.97      0.97      1000
        bird       0.96      0.94      0.95      1000

   micro avg       0.96      0.96      0.96      3000
   macro avg       0.96      0.96      0.96      3000
weighted avg       0.96      0.96      0.96      3000



In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from sklearn.metrics import classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.in_ch, self.out_ch, self.stride = in_ch, out_ch, stride

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        shortcut = x
        if self.stride != 1 or self.in_ch != self.out_ch:
            shortcut = x[:, :, ::self.stride, ::self.stride]
            pad = self.out_ch - self.in_ch
            shortcut = F.pad(shortcut, (0, 0, 0, 0, pad // 2, pad // 2))
        return F.relu(out + shortcut)

class ResNet20(nn.Module):
    def __init__(self, n=3):
        super().__init__()
        self.conv1  = nn.Conv2d(3, 16, 3, padding=1, bias=False)
        self.bn1    = nn.BatchNorm2d(16)
        self.group1 = self._group(16, 16, n, stride=1)
        self.group2 = self._group(16, 32, n, stride=2)
        self.group3 = self._group(32, 64, n, stride=2)
        self.pool   = nn.AdaptiveAvgPool2d(1)
        self.fc     = nn.Linear(64, 10)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

    def _group(self, in_ch, out_ch, n, stride):
        return nn.Sequential(
            ResidualBlock(in_ch, out_ch, stride),
            *[ResidualBlock(out_ch, out_ch) for _ in range(n - 1)]
        )

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.group1(x)
        x = self.group2(x)
        x = self.group3(x)
        return self.fc(self.pool(x).view(x.size(0), -1))

model = ResNet20().to(device)
print(f"Params: {sum(p.numel() for p in model.parameters()):,}  (paper: ~270k)")
print(f"Output: {model(torch.zeros(1, 3, 32, 32).to(device)).shape}")

train_tf = transforms.Compose([
    transforms.Pad(4), transforms.RandomCrop(32), transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])
test_tf = transforms.Compose([
    transforms.ToTensor(), transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])
trainloader = torch.utils.data.DataLoader(
    torchvision.datasets.CIFAR10('./data', train=True,  download=True, transform=train_tf), 128, shuffle=True,  num_workers=2)
testloader  = torch.utils.data.DataLoader(
    torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=test_tf),  128, shuffle=False, num_workers=2)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[82, 123], gamma=0.1)

for epoch in range(164):
    model.train()
    for x, y in trainloader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        criterion(model(x), y).backward()
        optimizer.step()
    scheduler.step()
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch + 1}/164")

model.eval()
correct, total, all_preds, all_targets = 0, 0, [], []
with torch.no_grad():
    for x, y in testloader:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(1)
        correct += pred.eq(y).sum().item()
        total += y.size(0)
        all_preds.extend(pred.cpu().numpy())
        all_targets.extend(y.cpu().numpy())

print(f"Top-1 Accuracy: {100 * correct / total:.2f}%")
print(classification_report(all_targets, all_preds, labels=[0, 1, 2],
                            target_names=['plane', 'car', 'bird'], zero_division=0))

Params: 269,722  (paper: ~270k)
Output: torch.Size([1, 10])


/home/suwaraich/CS_343_AI/.venv/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Epoch 20/164
Epoch 40/164
Epoch 60/164
Epoch 80/164
Epoch 100/164
Epoch 120/164
Epoch 140/164
Epoch 160/164
Top-1 Accuracy: 91.82%
              precision    recall  f1-score   support

       plane       0.92      0.92      0.92      1000
         car       0.96      0.97      0.96      1000
        bird       0.91      0.89      0.90      1000

   micro avg       0.93      0.93      0.93      3000
   macro avg       0.93      0.92      0.93      3000
weighted avg       0.93      0.93      0.93      3000

